<a href="https://colab.research.google.com/github/Lee-hosuk/game-agent/blob/main/2048/game_2048.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔢 2048 - 기대값 탐색(Expectimax) 에이전트

4×4 판에서 타일을 밀어 같은 숫자끼리 합치고, **2048 타일**을 만드는 게임입니다.

## 사목과 뭐가 다를까?
| | 사목 | 2048 |
|---|---|---|
| 상대 | 나를 이기려는 **상대** | 랜덤으로 나오는 **새 타일** |
| 상대 차례 계산 | 나에게 **가장 불리한** 수 (min) | 모든 경우의 **평균** (기대값) |
| 탐색 이름 | 미니맥스 / 알파-베타 | **Expectimax** |

새 타일은 나를 괴롭히려고 나오는 게 아니라 **운**입니다. 그래서 "최악의 경우"가 아니라 "평균적으로 가장 좋은 수"를 고릅니다.

## 1. 2048 게임 만들기
판은 16칸짜리 리스트입니다. `0`은 빈칸, 나머지는 타일 숫자예요.

In [ ]:
import random
import time

MOVES = ["up", "down", "left", "right"]
ARROW = {"up": "↑", "down": "↓", "left": "←", "right": "→"}

def spawn(b):
    """빈칸 하나에 새 타일을 놓음 (90% 확률로 2, 10% 확률로 4)"""
    empty = [i for i in range(16) if b[i] == 0]
    if empty:
        b[random.choice(empty)] = 2 if random.random() < 0.9 else 4

def new_board():
    b = [0] * 16
    spawn(b); spawn(b)
    return b

def merge_row(row):
    """한 줄을 왼쪽으로 밀기. (새 줄, 얻은 점수)를 돌려줌"""
    tiles = [x for x in row if x]           # 빈칸 빼고 모으기
    out, points, i = [], 0, 0
    while i < len(tiles):
        if i + 1 < len(tiles) and tiles[i] == tiles[i + 1]:
            out.append(tiles[i] * 2)          # 같은 숫자 두 개 → 합치기
            points += tiles[i] * 2
            i += 2
        else:
            out.append(tiles[i])
            i += 1
    return out + [0] * (4 - len(out)), points

# 방향마다 "벽 쪽부터" 읽을 칸 번호. 이렇게 하면 모든 방향을 merge_row 하나로 처리할 수 있음
LINES = {
    "left":  [[r * 4 + c for c in range(4)] for r in range(4)],
    "right": [[r * 4 + c for c in reversed(range(4))] for r in range(4)],
    "up":    [[r * 4 + c for r in range(4)] for c in range(4)],
    "down":  [[r * 4 + c for r in reversed(range(4))] for c in range(4)],
}

def move(b, direction):
    """(새 판, 얻은 점수, 움직였는지)를 돌려줌. 새 타일은 아직 안 놓음"""
    new = b[:]
    total = 0
    for line in LINES[direction]:
        merged, pts = merge_row([b[i] for i in line])
        total += pts
        for i, v in zip(line, merged):
            new[i] = v
    return new, total, new != b

def legal_moves(b):
    return [d for d in MOVES if move(b, d)[2]]

def show(b):
    for r in range(4):
        print(" ".join(f"{v:5d}" if v else "    ." for v in b[r*4:r*4+4]))

print(merge_row([2, 2, 0, 4]), merge_row([4, 4, 4, 4]), merge_row([2, 0, 2, 2]))
b = [2, 2, 0, 4,
     0, 8, 8, 0,
     4, 0, 4, 4,
     0, 0, 0, 2]
show(b); print("\n← 왼쪽으로 밀면:"); show(move(b, "left")[0])

## 2. 한 판 진행하는 함수와 기준 봇들
정책(policy)은 `판 → 방향`을 돌려주는 함수입니다.
- **랜덤 봇**: 아무 방향으로나 밂
- **점수 욕심 봇**: 지금 당장 점수를 가장 많이 얻는 방향으로 밂 (앞일은 생각 안 함)

In [ ]:
def play(policy, record=None, max_moves=100000):
    b = new_board()
    score = moves = 0
    if record is not None:
        record.append(b[:])
    while moves < max_moves and legal_moves(b):
        d = policy(b)
        b, pts, _ = move(b, d)
        score += pts
        moves += 1
        spawn(b)
        if record is not None:
            record.append(b[:])
    return {"max_tile": max(b), "score": score, "moves": moves, "board": b}

def random_bot(b):
    return random.choice(legal_moves(b))

def greedy_bot(b):
    return max(legal_moves(b), key=lambda d: move(b, d)[1])

random.seed(0)
for name, bot in [("랜덤", random_bot), ("점수 욕심", greedy_bot)]:
    r = play(bot)
    print(f"{name:<6} | 최고 타일 {r['max_tile']:5d} | 점수 {r['score']:6d} | {r['moves']}수")

## 3. 평가 함수: 사람의 전략을 점수로 바꾸기 ⭐

2048 고수들은 보통 이렇게 둡니다.
- 가장 큰 타일을 **한쪽 구석**에 붙여둔다
- 큰 타일부터 작은 타일 순서로 **뱀처럼 줄을 세운다** (그래야 연달아 합칠 수 있음)
- **빈칸을 많이** 남겨서 판이 막히지 않게 한다

이걸 그대로 점수로 만듭니다. 칸마다 **가중치**를 정하고, `타일 숫자 × 가중치`를 모두 더합니다.
뱀 모양 경로를 따라 가중치가 2배씩 커지도록 해서, 큰 타일이 왼쪽 위 구석부터 줄을 서면 점수가 높아집니다.

```
가중치 (2의 거듭제곱)        뱀 모양 경로
 2¹⁵  2¹⁴  2¹³  2¹²         →  →  →  ↓
 2⁸   2⁹   2¹⁰  2¹¹         ↓  ←  ←  ←
 2⁷   2⁶   2⁵   2⁴          →  →  →  ↓
 2⁰   2¹   2²   2³              ←  ←  ←
```
그리고 빈칸 하나마다 보너스(2¹²)를 더합니다.

In [ ]:
WEIGHTS = [
    2**15, 2**14, 2**13, 2**12,
    2**8,  2**9,  2**10, 2**11,
    2**7,  2**6,  2**5,  2**4,
    2**0,  2**1,  2**2,  2**3,
]
EMPTY_BONUS = 2**12

def evaluate(b):
    return sum(w * v for w, v in zip(WEIGHTS, b)) + b.count(0) * EMPTY_BONUS

good = [256, 128, 64, 32,
          4,   8, 16, 16,
          2,   0,  0,  0,
          0,   0,  0,  0]
bad  = [  4,  32,  2, 16,
        128,   0, 64,  8,
          0, 256,  0, 16,
          2,   0,  4,  0]
print("뱀 모양으로 정리된 판:", evaluate(good))
print("흩어진 판:          ", evaluate(bad), "  (같은 타일들인데 점수가 훨씬 낮음)")

## 4. Expectimax: 새 타일의 "평균"을 계산하기 ⭐

```
          [내 차례: 4방향 중 최고를 고름]       ← max
         ↑         ↓         ←         →
   [새 타일 차례: 모든 경우의 평균]               ← 기대값 (chance)
   빈칸 A에 2 (90%) / 4 (10%), 빈칸 B에 2 / 4, ...
              ↓
      [다시 내 차례] → [다시 새 타일] → ... → 평가 함수
```

- **내 차례 (max)**: 4방향 중 값이 가장 큰 방향
- **새 타일 차례 (chance)**: 빈칸마다 2가 나올 경우(90%)와 4가 나올 경우(10%)를 모두 계산해서 **확률로 가중 평균**

`depth`는 **몇 수 앞까지 내다보는지**입니다. (한 수 = 내가 밀기 + 새 타일 등장)
- `depth = 0`: 밀어본 결과를 평가 함수로만 판단 (탐색 없음)
- `depth = 1`: 밀고 → 새 타일이 나오는 모든 경우의 평균
- `depth = 2`: 밀고 → 새 타일 → 또 밀고 → 또 새 타일까지

In [ ]:
DEAD = -10**12      # 더 밀 수 없는 판 (게임 오버)
nodes = 0          # 살펴본 판의 개수

def max_value(b, depth):
    """내 차례: 가장 좋은 방향의 값"""
    global nodes
    nodes += 1
    if depth == 0:
        return evaluate(b)
    best = DEAD
    for d in MOVES:
        nb, _, moved = move(b, d)
        if moved:
            best = max(best, chance_value(nb, depth))
    return best

def chance_value(b, depth):
    """새 타일 차례: 모든 빈칸 × (2 또는 4)의 확률 가중 평균"""
    global nodes
    nodes += 1
    empty = [i for i in range(16) if b[i] == 0]
    total = 0
    for i in empty:
        for tile, p in ((2, 0.9), (4, 0.1)):
            b2 = b[:]
            b2[i] = tile
            total += p * max_value(b2, depth - 1)
    return total / len(empty)          # 빈칸 중 어디에 나올지는 똑같은 확률

def best_move(b, depth):
    best, best_d = None, None
    for d in MOVES:
        nb, _, moved = move(b, d)
        if moved:
            v = evaluate(nb) if depth == 0 else chance_value(nb, depth)
            if best is None or v > best:
                best, best_d = v, d
    return best_d

def make_expectimax_bot(depth):
    return lambda b: best_move(b, depth)

print("위 예시 판에서 깊이 2가 고른 방향:", ARROW[best_move(b, 2)])

## 5. 한 판 해보기
깊이 2로 한 판을 둡니다. 2048 이상까지 가면 **2,000수 넘게** 둬서 10~30초 걸려요.

In [ ]:
random.seed(1)
t0 = time.time()
r = play(make_expectimax_bot(2))
show(r["board"])
print(f"\n최고 타일 {r['max_tile']} | 점수 {r['score']} | {r['moves']}수 | {time.time()-t0:.1f}초")

## 6. 실험: 어떤 봇이 얼마나 갈까? 🏆
여러 판씩 두고 **최고 타일**이 어디까지 갔는지 비교합니다.

⏱️ 깊이 2 봇이 시간이 가장 오래 걸려서 전체 **1~3분** 정도 걸립니다.

In [ ]:
bots = {
    "random":        (random_bot, 20),
    "greedy score":  (greedy_bot, 20),
    "eval only (d0)": (make_expectimax_bot(0), 10),
    "expectimax d1": (make_expectimax_bot(1), 10),
    "expectimax d2": (make_expectimax_bot(2), 5),
}

random.seed(0)
results = {}
for name, (bot, n) in bots.items():
    t0 = time.time()
    tiles = [play(bot)["max_tile"] for _ in range(n)]
    results[name] = tiles
    print(f"{name:<15} | 최고 타일 {sorted(tiles)} | {time.time()-t0:.0f}초")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

targets = [256, 512, 1024, 2048, 4096]
names = list(results)
x = np.arange(len(names))
width = 0.16

plt.figure(figsize=(9, 4))
for k, t in enumerate(targets):
    rate = [100 * np.mean([m >= t for m in results[n]]) for n in names]
    plt.bar(x + (k - 2) * width, rate, width, label=f">= {t}")
plt.xticks(x, names)
plt.ylabel("% of games")
plt.title("2048: how far does each bot get?")
plt.legend()
plt.savefig("2048_results.png", dpi=100, bbox_inches="tight")
plt.show()

**볼 포인트**
- **점수 욕심 봇**은 당장 점수만 보니까 판이 금방 엉망이 됩니다.
- **평가 함수만** 써도(깊이 0) 꽤 좋아집니다. 사람의 전략을 점수로 바꾼 효과예요.
- **새 타일의 평균까지** 고려하면(깊이 1, 2) 판이 막히는 상황을 미리 피해서 훨씬 멀리 갑니다.

## 7. 탐색량 비교
깊이가 1 늘 때마다 살펴보는 판이 얼마나 늘어나는지 봅니다. 새 타일 경우의 수(빈칸 × 2)가 많아서 금방 커집니다.

In [ ]:
random.seed(5)
sample = new_board()
for _ in range(30):                       # 적당히 진행된 판 만들기
    if not legal_moves(sample): break
    sample = move(sample, best_move(sample, 1))[0]; spawn(sample)
show(sample)
print(f"빈칸 {sample.count(0)}개\n")
for depth in [0, 1, 2, 3]:
    nodes = 0; t0 = time.time()
    d = best_move(sample, depth)
    print(f"깊이 {depth}: {ARROW[d]}  | 살펴본 판 {nodes:>8,}개 | {time.time()-t0:.3f}초")

## 8. 경기 GIF 만들기 🎞️
깊이 2 봇의 한 판을 GIF로 저장합니다. 수가 많아서 몇 수마다 한 장씩만 씁니다.

In [ ]:
import imageio
from IPython.display import Image as IPImage, display
from PIL import Image, ImageDraw, ImageFont

TILE_COLORS = {0: "#cdc1b4", 2: "#eee4da", 4: "#ede0c8", 8: "#f2b179", 16: "#f59563", 32: "#f67c5f",
               64: "#f65e3b", 128: "#edcf72", 256: "#edcc61", 512: "#edc850", 1024: "#edc53f", 2048: "#edc22e", 4096: "#3c3a32"}

def load_font(size):
    try:
        return ImageFont.load_default(size=size)
    except TypeError:                      # 옛날 Pillow 버전
        return ImageFont.load_default()

def board_image(b, title="", cell=48):
    pad = 6
    W = pad + 4 * (cell + pad)
    img = Image.new("RGB", (W, W + 22), "#faf8ef")
    draw = ImageDraw.Draw(img)
    draw.text((pad, 4), title, fill="#776e65", font=load_font(13))
    draw.rectangle([0, 22, W, W + 22], fill="#bbada0")
    for i, v in enumerate(b):
        r, c = divmod(i, 4)
        x, y = pad + c * (cell + pad), 22 + pad + r * (cell + pad)
        draw.rectangle([x, y, x + cell, y + cell], fill=TILE_COLORS.get(v, "#3c3a32"))
        if v:
            font = load_font(20 if v < 100 else 16 if v < 1000 else 13)
            color = "#776e65" if v <= 4 else "white"
            draw.text((x + cell / 2, y + cell / 2), str(v), fill=color, font=font, anchor="mm")
    return np.array(img)

random.seed(1)
record = []
r = play(make_expectimax_bot(2), record=record)
step = max(1, len(record) // 60)          # 60장 안팎만 사용 (파일 크기 줄이기)
frames = [board_image(bd, f"move {i}", cell=40) for i, bd in enumerate(record) if i % step == 0]
frames.append(board_image(record[-1], f"max tile {r['max_tile']}  (score {r['score']})", cell=40))
durations = [150] * (len(frames) - 1) + [3000]     # 밀리초 단위. 마지막 장면은 3초
imageio.mimsave("2048_game.gif", frames, duration=durations, loop=0)
print(f"{len(record)}수 중 {len(frames)}장으로 GIF 저장")
display(IPImage(open("2048_game.gif", "rb").read()))

## 9. 직접 해보기 🎮
방향 버튼으로 직접 두고, 막히면 AI에게 물어보세요.
- **🤖 AI 추천**: 깊이 2 AI가 어느 방향을 추천하는지 알려줌
- **🤖 AI가 한 수**: AI가 대신 한 수 둠
- **⏩ AI 20수**: AI가 연속으로 20수 둠
- **🔄 새 게임**

In [ ]:
import ipywidgets as widgets
from IPython.display import display

def play_2048(ai_depth=2):
    state = {"b": new_board(), "score": 0}
    board_view = widgets.HTML()
    status = widgets.HTML()

    def render(msg=""):
        b = state["b"]
        cells = ""
        for v in b:
            color = TILE_COLORS.get(v, "#3c3a32")
            text = "#776e65" if v <= 4 else "white"
            size = 26 if v < 100 else 22 if v < 1000 else 18
            cells += (f"<div style='width:62px;height:62px;background:{color};color:{text};"
                      f"display:flex;align-items:center;justify-content:center;font:bold {size}px sans-serif;"
                      f"border-radius:4px'>{v if v else ''}</div>")
        board_view.value = (f"<div style='display:grid;grid-template-columns:repeat(4,62px);gap:6px;"
                            f"background:#bbada0;padding:6px;border-radius:6px;width:fit-content'>{cells}</div>")
        over = not legal_moves(b)
        head = f"<b>점수 {state['score']}</b> · 최고 타일 <b>{max(b)}</b>"
        if over:
            head += " · <b style='color:#c0392b'>게임 오버!</b>"
        status.value = f"<div style='font-size:16px'>{head}<br>{msg}</div>"

    def do_move(d):
        b = state["b"]
        nb, pts, moved = move(b, d)
        if not moved:
            return False
        state["score"] += pts
        spawn(nb)
        state["b"] = nb
        return True

    def on_dir(d):
        def handler(_):
            render("" if do_move(d) else f"{ARROW[d]} 방향으로는 움직일 수 없어요.")
        return handler

    def on_hint(_):
        if legal_moves(state["b"]):
            render(f"🤖 AI 추천: <b>{ARROW[best_move(state['b'], ai_depth)]}</b>")

    def on_ai(n):
        def handler(_):
            for _ in range(n):
                if not legal_moves(state["b"]):
                    break
                do_move(best_move(state["b"], ai_depth))
            render(f"🤖 AI가 {n}수 뒀어요." if n > 1 else "🤖 AI가 한 수 뒀어요.")
        return handler

    def on_new(_):
        state["b"], state["score"] = new_board(), 0
        render("새 게임!")

    def btn(label, cb, width="56px"):
        b = widgets.Button(description=label, layout=widgets.Layout(width=width))
        b.on_click(cb)
        return b

    pad = widgets.VBox([
        widgets.HBox([widgets.Label(layout=widgets.Layout(width="56px")), btn("↑", on_dir("up"))]),
        widgets.HBox([btn("←", on_dir("left")), btn("↓", on_dir("down")), btn("→", on_dir("right"))]),
    ])
    tools = widgets.VBox([btn("🤖 AI 추천", on_hint, "120px"), btn("🤖 AI가 한 수", on_ai(1), "120px"),
                          btn("⏩ AI 20수", on_ai(20), "120px"), btn("🔄 새 게임", on_new, "120px")])
    display(widgets.VBox([status, widgets.HBox([board_view, widgets.HTML("&nbsp;&nbsp;"), widgets.VBox([pad, tools])])]))
    render("방향 버튼을 눌러 시작하세요.")

play_2048(ai_depth=2)

## 🔥 도전 과제
1. **가중치 바꾸기**: `WEIGHTS`를 2배 대신 4배씩 커지게 하거나, `EMPTY_BONUS`를 바꾸면 어떻게 될까요? 6번 실험으로 비교해보세요.
2. **새로운 평가 요소**: "옆 칸과 숫자가 비슷할수록 +"(합치기 쉬움), "줄이 한 방향으로 정렬돼 있을수록 +" 같은 점수를 더해보세요.
3. **더 깊이**: 깊이 3은 훨씬 강하지만 매우 느립니다. 빈칸이 많을 때는 몇 칸만 골라서 계산하는 식으로 빠르게 만들어보세요.
4. **4096 도전**: 어떤 설정이 4096 타일을 가장 자주 만드나요?
5. **강화학습과 비교**: 2048은 강화학습(n-tuple network, TD 학습)으로도 아주 잘 풀립니다. 평가 함수를 사람이 만들지 않고 **학습시킨다면** 어떻게 해야 할까요?